%md
# 01 - Bronze : ingestion des données Vélib'
**Objectif :** charger les fichiers JSON du volume `raw` dans des tables Delta, avec **Auto Loader**.

- Une ligne par station (on déplie la liste `stations`)
- On garde les valeurs brutes, sans nettoyage
- On ajoute le nom du fichier source et l'heure de chargement

In [0]:
%sql
-- Dossier où Auto Loader note les fichiers déjà traités
CREATE VOLUME IF NOT EXISTS workspace.velib_bronze.checkpoints;

## 1. Fonction de chargement avec Auto Loader
Auto Loader ne lit que les **nouveaux fichiers** à chaque exécution. Il garde la liste des fichiers déjà lus dans le dossier `checkpoints`.

In [0]:
%python
from pyspark.sql import functions as F

SOURCE = "/Volumes/workspace/velib_bronze/raw"
CHECKPOINTS = "/Volumes/workspace/velib_bronze/checkpoints"

def charger_bronze(type_fichier):
    """type_fichier = 'station_status' ou 'station_information'"""
    checkpoint = f"{CHECKPOINTS}/{type_fichier}"

    # Lecture des fichiers JSON avec Auto Loader
    df_raw = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", checkpoint)     # où il garde le schéma
        .option("cloudFiles.inferColumnTypes", "true")       # vrais types (sinon tout en texte)
        .option("multiLine", "true")
        .option("pathGlobFilter", f"{type_fichier}_*.json")  # seulement ce type de fichier
        .load(SOURCE))

    # Une ligne par station + colonnes techniques
    df_bronze = (df_raw
        .select(
            F.explode("data.stations").alias("station"),
            F.col("lastUpdatedOther").alias("last_updated_epoch"),
            F.col("_metadata.file_path").alias("source_file"),
            F.current_timestamp().alias("ingestion_time"))
        .select("station.*", "last_updated_epoch", "source_file", "ingestion_time"))

    # Écriture dans une table Delta
    query = (df_bronze.writeStream
        .option("checkpointLocation", checkpoint)
        .trigger(availableNow=True)   # traite les nouveaux fichiers puis s'arrête
        .toTable(f"workspace.velib_bronze.{type_fichier}"))

    query.awaitTermination()  # attend la fin avant de passer à la suite
    print(f"Table velib_bronze.{type_fichier} à jour")

## 2. Chargement des deux types de fichiers

On lance la fonction pour chaque type de fichier du volume `raw` :
- **`station_status`** : disponibilité des stations à un instant T (vélos mécaniques et électriques dispos, places libres). Résultat dans la table `velib_bronze.station_status`.
- **`station_information`** : infos fixes des stations (nom, coordonnées GPS, capacité). Résultat dans la table `velib_bronze.station_information`.

Seuls les fichiers **pas encore chargés** sont traités.

In [0]:
%python
charger_bronze("station_status")
charger_bronze("station_information")

Table velib_bronze.station_status à jour
Table velib_bronze.station_information à jour


## 3. Vérification
On vérifie que le chargement s'est bien passé, en comptant le nombre de stations chargées **pour chaque fichier source**.

Résultat attendu :
- une ligne par fichier `station_status` chargé
- environ 1 500 stations par fichier
- si on relance le chargement, les chiffres ne doivent **pas doubler** (Auto Loader ne recharge pas les fichiers déjà traités)

In [0]:
%sql
-- Nombre de stations chargées par fichier source
-- (1 ligne = 1 fichier, environ 1 500 stations attendues)
SELECT
  source_file,                         -- fichier JSON d'origine
  COUNT(*)            AS nb_stations,  -- nombre de stations dans ce fichier
  MIN(ingestion_time) AS charge_le     -- date et heure du chargement
FROM workspace.velib_bronze.station_status
GROUP BY source_file;

source_file,nb_stations,charge_le
/Volumes/workspace/velib_bronze/raw/station_status_2026-10-24_1852.json,1519,2026-10-24T17:09:30.341Z
